In [2]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


In [3]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- Text can be converted to bytearray
- casting to a list give the list of bytes that is integer

In [4]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- Number of id could be different because some chars need more bytes to represent them
- Bytes value range is 0-255 ($2^8$)

In [5]:
text = "é"
print(list(bytearray(text, encoding="utf-8")))

[195, 169]


The algorithm
- Find most frequent adjacent pair of bytes (a, b)
- Replace the pair (a, b) with new ID
- Repeat until no gain
- Decoding: reverse the process by substituating the id by the correspond pair

- 0-255 is reserved for all possible single bytes
- Each iteration, new_id = 256 + i, i=0, i=total_iter

`text = 'the cat in the hat`
Iteration 1: `[('t', 'h'), ('e', ' '),... ('t', 'h'), ('e', ' '), ('a', 't')]`
new id: 256 is associated to `('t', 'h')`
The sequence becomes:  `[('256', 'e'), (' ', 'c'),... ('256', 'e'), (' ', 'h'), ('a', 't')]`

In [6]:
text = "the cat in the hat"
text_bytes = list(bytearray(text, encoding="utf-8"))
text_bytes

[116,
 104,
 101,
 32,
 99,
 97,
 116,
 32,
 105,
 110,
 32,
 116,
 104,
 101,
 32,
 104,
 97,
 116]

- text: `the cat in the hat`

In [7]:
from collections import Counter
from typing import Sequence

def get_pair_count(ids: Sequence[int]) -> Counter:
    pairs = zip(ids, ids[1:])
    return Counter(pairs)

In [8]:

counts = get_pair_count(text_bytes)

In [9]:
counts

Counter({(116, 104): 2,
         (104, 101): 2,
         (101, 32): 2,
         (97, 116): 2,
         (32, 99): 1,
         (99, 97): 1,
         (116, 32): 1,
         (32, 105): 1,
         (105, 110): 1,
         (110, 32): 1,
         (32, 116): 1,
         (32, 104): 1,
         (104, 97): 1})

In [10]:
top_pair = counts.most_common(1)[0]
top_pair

((116, 104), 2)

- Iterate overall
- The pair is replaced by the new id and other is kept as is
- 

In [11]:
def merge(ids: Sequence[int], pair: tuple[int, int], new_id: int) -> list[int]:
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


The loop

In [12]:

def train(text: str, vocab_size: int = 260) -> tuple:
    #TODO: need the correspondance between char and char index
    # Every number in the tuple correspond to a char and I need to track it
    # The base vocab is ASCII char
    merges: dict[tuple[int, int], int]= {}
    unique_chars = [chr(i) for i in range(256)]
    # We need to add chars that is not in unique_chars like emoji and etc
    unique_chars.extend([char for char in sorted(set(text)) if char not in unique_chars])
    int2token = {i: char for i, char in enumerate(unique_chars)}
    token2int = {char: i for i, char in int2token.items()}
    token_ids = [token2int[char] for char in text]
    for i in range(vocab_size-256):
        counts = get_pair_count(ids=token_ids)
        if not counts:
            break
        top_pair = max(counts, key=counts.get)
        new_id = 256 + i
        token_ids = merge(ids=token_ids, pair=top_pair, new_id=new_id)
        merges[top_pair] = new_id
    for (p1, p2), token_id in merges.items():
        token = int2token[p1] + int2token[p2]
        int2token[token_id] = token
        token2int[token] = token_id
    return merges, int2token, token2int


In [13]:
merges, vocab, inverse_vcab = train(text=text, vocab_size=260)

In [14]:
def decode(ids: list[int], vocab: dict[int, str]):
    words = []
    for id in ids:
        if id not in vocab:
            raise ValueError
        words.append(vocab[id])
    return words

In [15]:
decode(ids=[256, 257], vocab=vocab)

['th', 'the']

In [16]:
def encode(text: str, merges: dict[tuple[int, int], int], token2int: dict[str, int]) -> list[int]:
    ids = [token2int[char] for char in text]
    for pair, new_id in merges.items():
        ids = merge(ids, pair, new_id)
    return ids

In [17]:
encode(text="the cat", merges=merges, token2int=inverse_vcab)

[258, 99, 259]

- Need to track merge str and concat them
- To decode, we take a list of int in the vocab and return the corresponding string
- After the train, how to encode new text?

Two methods to encode:
- Iterate over merge rules as I do. But expensive because we need to iterate over full pass
- Scan adjacent pairs in each iteration, find lower rank and merge that pair everywhere. This makes fewer pass for a specific input


## Reference

- https://sebastianraschka.com/blog/2025/bpe-from-scratch.html